In [ ]:
using Plots

In [ ]:
# ------------------------------------------------------------
# Fourier-Lagrange basis function ℓ_j(x)
# N must be odd: N = 2n+1
# ------------------------------------------------------------
function fourier_lagrange_basis(j, x, N)

    xj = 2π * j / N
    θ = x - xj
    # abvoid small divsors
    if abs(sin(θ/2)) < 1e-14
        return 1.0
    else
        return sinc(N * θ/(2*π))/sinc(θ/(2*π))
    end
end


In [ ]:
n = 13
N = 2 * n + 1
x = LinRange(0, 2π, 500)
x_nodes = 2π * (0:N-1) / N


plot(x, fourier_lagrange_basis.(0, x, N), label="ℓ₀")
plot!(x, fourier_lagrange_basis.(1, x, N), label="ℓ₁")
plot!(x, fourier_lagrange_basis.(2, x, N), label="ℓ₂")
plot!(x_nodes, ones(N), ls=:dash, color = :black, label="")
scatter!(x_nodes, zeros(N), label="")
xlabel!("x")


In [ ]:
using FFTW

In [ ]:
N = 16;
x = LinRange(0, 2π, N+1)[1:end-1];
f = cos.(x);
fhat = fft(f)

In [ ]:
N = 16;
x = LinRange(0, 2π, N+1)[1:end-1];
f = @. 1 + sin(x) + cos(3x);
fhat = fft(f)

In [ ]:
round.(fhat, digits=3)

In [ ]:
N = 16;
fhat = zeros(ComplexF64, N);
fhat[2] = N/2;
fhat[end] = N/2;
f = ifft(fhat)

In [ ]:
using Plots

In [ ]:
x = LinRange(0, 2π, N+1)[1:end-1];
plot(x, real.(f), label="f")
scatter!(x, cos.(x), label="cos(x)")
xlabel!("x")
savefig("fourier_cos.pdf")

In [ ]:
N = 16;
x = LinRange(0, 2π, N+1)[1:end-1];
f = @. 1 + sin(x) + cos(3x);
fhat = fft(f)

In [ ]:
N = 16;
x = LinRange(0, 2π, N+1)[1:end-1];
f = cos.(x);
fhat = fft(f)
fftshift(fhat)

In [ ]:
using LinearAlgebra

In [ ]:
N = 16;
x = LinRange(0, 2π, N+1)[1:end-1];
k = [0:N÷2; -N÷2+1:-1];
ik = im * k;

f = cos.(x);
fhat = fft(f)
ikfhat = ik .* fhat
g = ifft(ikfhat);
norm(g - (-sin.(x)),Inf)

In [ ]:
N_vals = [4, 8, 16, 32, 64, 128, 256, 512, 1024]
errors = zeros(length(N_vals))

for (i, N) in enumerate(N_vals)
    x = LinRange(0, 2π, N+1)[1:end-1];
    k = [0:N÷2; -N÷2+1:-1];
    ik = im * k;
    f = @. exp(sin(x));
    fhat = fft(f)
    ikfhat = ik .* fhat
    g = ifft(ikfhat);
    fp_exact = @. exp(sin(x)) * cos(x);
    errors[i] = norm(g - fp_exact, Inf);
end

scatter(N_vals, errors, xscale=:log2, yscale=:log10, label="Error", marker=:o)
xticks!(N_vals)
yticks!(10.0 .^(-14:2:0))
xlabel!("N")
ylabel!("Infinity Norm of Error")
savefig("fourier_error.pdf")

In [ ]:
2^5

In [ ]:
N = 32;
x = LinRange(0, 2π, N+1)[1:end-1];
k = [0:N÷2; -N÷2+1:-1];
ik = im * k;

u_exact = @.cos(sin(x));

# analytically compute this
f = @. -sin(x) * sin(sin(x)) + cos(x)^2 * cos(sin(x));

fhat = fft(f)
uhat = zeros(ComplexF64, N);

@. uhat[2:end] = fhat[2:end] / k[2:end]^2;
u = ifft(uhat);

# shift solutions to make for fair comparison
plot(x, real.(u .- u[1]), label="Numerical Solution")
scatter!(x, u_exact  .-u_exact[1], label="Exact Solution")
xlabel!("x")
savefig("fourier_poisson.pdf")

In [ ]:
N = 32;
x = LinRange(0, 2π, N+1)[1:end-1];
k = [0:N÷2; -N÷2+1:-1];
ik = im * k;

u_exact = @. cos(sin(x));

# analytically compute u - u'' = f
f = @. cos(sin(x)) - sin(x) * sin(sin(x)) + cos(x)^2 * cos(sin(x));

fhat = fft(f)
uhat = zeros(ComplexF64, N);

@. uhat = fhat / (1 + k^2);

u = ifft(uhat);

# shift solutions to make for fair comparison
plot(x, real.(u .- u[1]), label="Numerical Solution")
scatter!(x, u_exact  .-u_exact[1], label="Exact Solution")
xlabel!("x")
# savefig("fourier_poisson.pdf")